In [2]:
# --- Imports ---

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# --- Constants ---

PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

In [5]:
# --- Data fetching report ---

report = pd.read_csv(INTERIM_DIR / "collection_report.csv")

In [6]:
report

,dataset,code,level,nuts0,nuts1,nuts2,year_min,year_max,n_years,shape,missing_pct,notes
0,aerobic_activity,hlth_ehis_pe2e,all,32,0,0,2014,2019,2,"(32, 2)",7.8,% meeting 150+ min/week. 2014 and 2019 only.
1,family_contact,ilc_scp09,all,36,0,0,2015,2022,2,"(36, 2)",8.3,% getting together with family at least weekly...
2,gdp_per_capita,nama_10r_2gdp,all,33,110,285,2000,2024,25,"(428, 25)",3.1,GDP per capita in PPS. NUTS0-2 confirmed.
3,heavy_drinking,hlth_ehis_al3e,all,32,0,0,2014,2019,2,"(32, 2)",7.8,% drinking heavily at least once/week. 2014 an...
4,internet_usage,isoc_r_iuse_i,all,38,126,225,2006,2025,20,"(389, 20)",36.2,% regularly using internet. NUTS2 confirmed.
5,life_expectancy,demo_r_mlifexp,all,37,125,345,1990,2024,35,"(507, 35)",24.3,Life expectancy at birth. No total sex — use F...
6,life_satisfaction,ilc_pw01,all,37,0,0,2013,2025,7,"(37, 7)",10.0,Meaning of life
7,obesity_rate,sdg_02_10,all,37,0,0,2008,2025,6,"(37, 6)",19.8,% population BMI>=30.
8,poverty_risk,ilc_li41,all,38,100,283,2003,2025,23,"(421, 23)",44.8,No sub-dimensions — just filter unit. NUTS2 co...
9,smoking,hlth_ehis_sk3e,all,32,0,0,2014,2019,2,"(32, 2)",3.1,% smokers total (light + heavy). 2014 and 2019...


In [13]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import linregress

PROJECT_ROOT  = Path.cwd().parent
INTERIM_DIR   = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

YEAR_START = 2013
YEAR_END   = 2024
WINDOW     = list(range(YEAR_START, YEAR_END + 1))

DATASETS = [
    "life_expectancy",
    "unemployment_rate",
    "poverty_risk",
    "internet_usage",
    "gdp_per_capita",
    "social_trust",
    "life_satisfaction",
    "aerobic_activity",
    "family_contact",
    "heavy_drinking",
    "obesity_rate",
    "smoking",
    "social_contact",
    "social_support",
]

# Load all interim datasets
interim = {}
for name in DATASETS:
    path = INTERIM_DIR / f"{name}.parquet"
    if path.exists():
        interim[name] = pd.read_parquet(path)
        print(f"✅ {name}: {interim[name].shape}")
    else:
        print(f"⚠️  {name}: not found")

✅ life_expectancy: (507, 35)
✅ unemployment_rate: (510, 27)
✅ poverty_risk: (421, 23)
✅ internet_usage: (389, 20)
✅ gdp_per_capita: (428, 25)
✅ social_trust: (37, 7)
✅ life_satisfaction: (37, 7)
✅ aerobic_activity: (32, 2)
✅ family_contact: (36, 2)
✅ heavy_drinking: (32, 2)
✅ obesity_rate: (37, 6)
✅ smoking: (32, 2)
✅ social_contact: (33, 2)
✅ social_support: (33, 1)


In [14]:
# Keep only years within window, drop years outside
windowed = {}
for name, df in interim.items():
    cols_in_window = [y for y in WINDOW if y in df.columns]
    windowed[name] = df[cols_in_window].copy()
    print(f"✅ {name}: {windowed[name].shape} | years: {cols_in_window[0]}–{cols_in_window[-1]}")

✅ life_expectancy: (507, 12) | years: 2013–2024
✅ unemployment_rate: (510, 12) | years: 2013–2024
✅ poverty_risk: (421, 12) | years: 2013–2024
✅ internet_usage: (389, 12) | years: 2013–2024
✅ gdp_per_capita: (428, 12) | years: 2013–2024
✅ social_trust: (37, 6) | years: 2013–2024
✅ life_satisfaction: (37, 6) | years: 2013–2024
✅ aerobic_activity: (32, 2) | years: 2014–2019
✅ family_contact: (36, 2) | years: 2015–2022
✅ heavy_drinking: (32, 2) | years: 2014–2019
✅ obesity_rate: (37, 4) | years: 2014–2022
✅ smoking: (32, 2) | years: 2014–2019
✅ social_contact: (33, 2) | years: 2013–2015
✅ social_support: (33, 1) | years: 2015–2015


In [15]:
print(f"{'dataset':<20} {'regions':>8} {'years':>6} {'missing%':>10} {'notes'}")
print("-" * 65)
for name, df in windowed.items():
    total_cells = df.size
    missing     = df.isnull().sum().sum()
    missing_pct = missing / total_cells * 100 if total_cells > 0 else float("nan")
    n_years     = df.shape[1]
    n_regions   = df.shape[0]
    note = ""
    if n_years <= 2:
        note = "⚠️  only 2 time points — static feature only"
    elif n_years == 1:
        note = "⚠️  single time point"
    elif missing_pct > 30:
        note = "⚠️  high missingness"
    print(f"{name:<20} {n_regions:>8} {n_years:>6} {missing_pct:>9.1f}%  {note}")

dataset               regions  years   missing% notes
-----------------------------------------------------------------
life_expectancy           507     12      10.2%  
unemployment_rate         510     12       9.8%  
poverty_risk              421     12      31.5%  ⚠️  high missingness
internet_usage            389     12      16.3%  
gdp_per_capita            428     12       0.7%  
social_trust               37      6       9.0%  
life_satisfaction          37      6       8.6%  
aerobic_activity           32      2       7.8%  ⚠️  only 2 time points — static feature only
family_contact             36      2       8.3%  ⚠️  only 2 time points — static feature only
heavy_drinking             32      2       7.8%  ⚠️  only 2 time points — static feature only
obesity_rate               37      4      12.2%  
smoking                    32      2       3.1%  ⚠️  only 2 time points — static feature only
social_contact             33      2       1.5%  ⚠️  only 2 time points — static fea

In [ ]:
def impute(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Linear interpolation for gaps <= 2 consecutive years
    2. Country mean for remaining gaps
    """
    # Step 1 — interpolate isolated gaps
    df = df.interpolate(axis=1, limit=2, limit_direction="both")

    # Step 2 — country mean for larger gaps
    # country code = first 2 chars of NUTS geo index
    df["country"] = df.index.str[:2]
    for year in df.columns[:-1]:  # exclude 'country'
        country_mean = df.groupby("country")[year].transform("mean")
        df[year] = df[year].fillna(country_mean)
    df = df.drop(columns=["country"])

    return df

In [17]:
PCA_FEATURES = [
    "unemployment_rate", "poverty_risk",
    "internet_usage", "gdp_per_capita",
    "social_trust", "life_satisfaction",
]

imputed = {}
for name in PCA_FEATURES:
    before = windowed[name].isnull().sum().sum()
    imputed[name] = impute(windowed[name].copy())
    after  = imputed[name].isnull().sum().sum()
    print(f"✅ {name}: {before} missing → {after} missing")

✅ unemployment_rate: 598 missing → 186 missing
✅ poverty_risk: 1592 missing → 41 missing
✅ internet_usage: 759 missing → 41 missing
✅ gdp_per_capita: 35 missing → 8 missing
✅ social_trust: 20 missing → 6 missing
✅ life_satisfaction: 19 missing → 6 missing


In [18]:
# Check which regions still have missing values after imputation
for name in PCA_FEATURES:
    df = imputed[name]
    still_missing = df[df.isnull().any(axis=1)]
    if len(still_missing) > 0:
        print(f"\n── {name} ({len(still_missing)} regions still missing) ──")
        print(still_missing.isnull().sum(axis=1).sort_values(ascending=False))


── unemployment_rate (62 regions still missing) ──
geo
BA      6
UKC     3
UK      3
UKD     3
UKC2    3
       ..
UKM7    3
UKN0    3
ME      2
ME00    2
ME0     2
Length: 62, dtype: int64

── poverty_risk (11 regions still missing) ──
geo
BA0     7
BA      7
XK0     6
XK00    6
XK      6
UK      4
AL      1
AL01    1
AL0     1
AL02    1
AL03    1
dtype: int64

── internet_usage (21 regions still missing) ──
geo
AL      3
BA      3
UK      2
ME0     2
ME      2
UKH     2
UKI     2
UKJ     2
UKC     2
UKD     2
UKE     2
UKF     2
UKG     2
UKL     2
UKM     2
UKN     2
UKK     2
XK      2
IS00    1
IS      1
IS0     1
dtype: int64

── gdp_per_capita (8 regions still missing) ──
geo
NO      1
NO0     1
NO02    1
NO06    1
NO07    1
NO08    1
NO09    1
NO0A    1
dtype: int64

── social_trust (3 regions still missing) ──
geo
IS    2
UK    2
XK    2
dtype: int64

── life_satisfaction (3 regions still missing) ──
geo
IS    2
UK    2
XK    2
dtype: int64


In [19]:
with open(PROJECT_ROOT / "data" / "interim" / "remaining_missing.txt", "w") as f:
    for name in PCA_FEATURES:
        df = imputed[name]
        still_missing = df[df.isnull().any(axis=1)]
        if len(still_missing) > 0:
            f.write(f"\n── {name} ({len(still_missing)} regions) ──\n")
            f.write(still_missing.isnull().sum(axis=1).sort_values(ascending=False).to_string())
            f.write("\n")

print("Saved — open data/interim/remaining_missing.txt")

Saved — open data/interim/remaining_missing.txt


In [20]:
balkans = ["BA", "BA0", "BA00", "XK", "XK0", "XK00", "AL", "AL0", "AL01", "AL02", "AL03", "ME", "ME0", "ME00"]

for name in PCA_FEATURES:
    df = imputed[name]
    balkan_rows = df[df.index.isin(balkans)]
    if len(balkan_rows) > 0:
        missing = balkan_rows.isnull().sum(axis=1)
        missing = missing[missing > 0]
        if len(missing) > 0:
            print(f"\n── {name} ──")
            print(missing)


── unemployment_rate ──
geo
BA      6
ME      2
ME0     2
ME00    2
dtype: int64

── poverty_risk ──
geo
AL      1
AL0     1
AL01    1
AL02    1
AL03    1
BA      7
BA0     7
XK      6
XK0     6
XK00    6
dtype: int64

── internet_usage ──
geo
AL     3
BA     3
ME     2
ME0    2
XK     2
dtype: int64

── social_trust ──
geo
XK    2
dtype: int64

── life_satisfaction ──
geo
XK    2
dtype: int64


In [22]:
with open(PROJECT_ROOT / "data" / "interim" / "balkan_missing.txt", "w") as f:
    for name in PCA_FEATURES:
        df = imputed[name]
        balkan_rows = df[df.index.isin(balkans)]
        if len(balkan_rows) > 0:
            missing = balkan_rows.isnull().sum(axis=1)
            missing = missing[missing > 0]
            if len(missing) > 0:
                f.write(f"\n── {name} ──\n")
                f.write(missing.to_string())
                f.write("\n")

print("Saved — open data/interim/balkan_missing.txt")

Saved — open data/interim/balkan_missing.txt


In [23]:
# Drop BA and XK from all PCA feature datasets
DROP_REGIONS = ["BA", "BA0", "BA00", "XK", "XK0", "XK00"]

for name in PCA_FEATURES:
    before = len(imputed[name])
    imputed[name] = imputed[name][~imputed[name].index.isin(DROP_REGIONS)]
    after = len(imputed[name])
    if before != after:
        print(f"✅ {name}: dropped {before - after} regions")

✅ unemployment_rate: dropped 1 regions
✅ poverty_risk: dropped 5 regions
✅ internet_usage: dropped 2 regions
✅ social_trust: dropped 1 regions
✅ life_satisfaction: dropped 1 regions


In [24]:
for name in PCA_FEATURES:
    imputed[name] = imputed[name].ffill(axis=1).bfill(axis=1)
    remaining = imputed[name].isnull().sum().sum()
    print(f"✅ {name}: {remaining} missing after ffill/bfill")

✅ unemployment_rate: 0 missing after ffill/bfill
✅ poverty_risk: 0 missing after ffill/bfill
✅ internet_usage: 0 missing after ffill/bfill
✅ gdp_per_capita: 0 missing after ffill/bfill
✅ social_trust: 0 missing after ffill/bfill
✅ life_satisfaction: 0 missing after ffill/bfill


In [25]:
print(f"{'dataset':<20} {'regions':>8} {'nuts0':>6} {'nuts1':>6} {'nuts2':>6}")
print("-" * 50)
for name in PCA_FEATURES:
    df = imputed[name]
    level_counts = df.index.str.len().value_counts().sort_index().to_dict()
    print(f"{name:<20} {len(df):>8} {level_counts.get(2,0):>6} {level_counts.get(3,0):>6} {level_counts.get(4,0):>6}")

dataset               regions  nuts0  nuts1  nuts2
--------------------------------------------------
unemployment_rate         509     35    123    351
poverty_risk              416     36     98    282
internet_usage            387     36    126    225
gdp_per_capita            428     33    110    285
social_trust               36     36      0      0
life_satisfaction          36     36      0      0


In [26]:
# Impute life expectancy — validation only, not PCA
imputed["life_expectancy"] = impute(windowed["life_expectancy"].copy())
imputed["life_expectancy"] = imputed["life_expectancy"].ffill(axis=1).bfill(axis=1)

remaining = imputed["life_expectancy"].isnull().sum().sum()
level_counts = imputed["life_expectancy"].index.str.len().value_counts().sort_index().to_dict()
print(f"life_expectancy: {remaining} missing | regions: {len(imputed['life_expectancy'])} | nuts0={level_counts.get(2,0)} nuts1={level_counts.get(3,0)} nuts2={level_counts.get(4,0)}")

life_expectancy: 0 missing | regions: 507 | nuts0=37 nuts1=125 nuts2=345


In [ ]:
def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute mean and OLS slope per region across the time window.
    Returns dataframe with two columns: {name}_mean and {name}_slope.
    Slope = change per year (positive = improving over time).
    """
    years = np.array(df.columns, dtype=float)

    means  = df.mean(axis=1)
    slopes = df.apply(
        lambda row: linregress(years, row.values)[0], axis=1
    )
    return pd.DataFrame({"mean": means, "slope": slopes}, index=df.index)


features = {}
for name in PCA_FEATURES:
    feat = compute_features(imputed[name])
    feat.columns = [f"{name}_mean", f"{name}_slope"]
    features[name] = feat
    print(f"✅ {name}: {feat.shape} | mean range [{feat.iloc[:,0].min():.2f}, {feat.iloc[:,0].max():.2f}]")

✅ unemployment_rate: (509, 2) | mean range [2.08, 28.25]
✅ poverty_risk: (416, 2) | mean range [4.42, 39.57]
✅ internet_usage: (387, 2) | mean range [43.54, 96.09]
✅ gdp_per_capita: (428, 2) | mean range [24.42, 261.83]
✅ social_trust: (36, 2) | mean range [3.43, 7.28]
✅ life_satisfaction: (36, 2) | mean range [5.60, 8.08]


In [ ]:
def broadcast_to_regions(country_df: pd.DataFrame,
                          target_index: pd.Index) -> pd.DataFrame:
    """
    Assign country-level values to all NUTS1/NUTS2 regions
    by matching the first 2 characters of the geo code.
    """
    country_codes = country_df.index.str[:2]
    result = country_df.copy()
    result.index = country_codes

    # Map each region in target_index to its country value
    target_countries = target_index.str[:2]
    return result.reindex(target_countries).set_index(target_index)


# Use unemployment_rate regions as the reference index (broadest NUTS2 coverage)
reference_index = imputed["unemployment_rate"].index

for name in ["social_trust", "life_satisfaction"]:
    before = len(features[name])
    features[name] = broadcast_to_regions(features[name], reference_index)
    after  = len(features[name])
    print(f"✅ {name}: {before} countries → {after} regions")

✅ social_trust: 36 countries → 509 regions
✅ life_satisfaction: 36 countries → 509 regions


In [29]:
# Join all PCA features on common index
feature_matrix = pd.concat(features.values(), axis=1, join="inner")

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Features: {feature_matrix.columns.tolist()}")
print(f"\nMissing values: {feature_matrix.isnull().sum().sum()}")

# NUTS level breakdown
level_counts = feature_matrix.index.str.len().value_counts().sort_index().to_dict()
print(f"\nnuts0={level_counts.get(2,0)} nuts1={level_counts.get(3,0)} nuts2={level_counts.get(4,0)}")

feature_matrix.head()


Feature matrix shape: (325, 12)
Features: ['unemployment_rate_mean', 'unemployment_rate_slope', 'poverty_risk_mean', 'poverty_risk_slope', 'internet_usage_mean', 'internet_usage_slope', 'gdp_per_capita_mean', 'gdp_per_capita_slope', 'social_trust_mean', 'social_trust_slope', 'life_satisfaction_mean', 'life_satisfaction_slope']

Missing values: 0

nuts0=32 nuts1=93 nuts2=200


,unemployment_rate_mean,unemployment_rate_slope,poverty_risk_mean,poverty_risk_slope,internet_usage_mean,internet_usage_slope,gdp_per_capita_mean,gdp_per_capita_slope,social_trust_mean,social_trust_slope,life_satisfaction_mean,life_satisfaction_slope
geo,,,,,,,,,,,,
AT,5.333333,-0.042657,14.258333,0.045105,75.840833,2.047168,124.750000,-0.982517,5.766667,-0.021328,7.833333,-0.013682
AT1,7.300000,-0.050350,15.386667,0.407133,78.349167,1.918147,123.416667,-1.311189,5.766667,-0.021328,7.833333,-0.013682
AT11,4.725000,-0.009441,10.075000,-0.315501,71.393750,2.047080,87.333333,-0.398601,5.766667,-0.021328,7.833333,-0.013682
AT12,4.566667,-0.095105,10.908333,0.148135,74.790417,2.286311,103.250000,-0.821678,5.766667,-0.021328,7.833333,-0.013682
AT13,10.158333,-0.031818,21.808333,0.223660,82.208333,1.644895,147.083333,-2.038462,5.766667,-0.021328,7.833333,-0.013682


In [30]:
# What regions are in unemployment but not in the feature matrix?
lost = imputed["unemployment_rate"].index.difference(feature_matrix.index)
lost_levels = lost.str.len().value_counts().sort_index().to_dict()
print(f"Lost regions: {len(lost)}")
print(f"nuts0={lost_levels.get(2,0)} nuts1={lost_levels.get(3,0)} nuts2={lost_levels.get(4,0)}")
print("\nSample:")
print(lost[:20].tolist())

Lost regions: 184
nuts0=3 nuts1=30 nuts2=151

Sample:
['CH', 'CH0', 'CH01', 'CH02', 'CH03', 'CH04', 'CH05', 'CH06', 'CH07', 'DE11', 'DE12', 'DE13', 'DE14', 'DE21', 'DE22', 'DE23', 'DE24', 'DE25', 'DE26', 'DE27']


In [31]:
# Check region coverage per dataset
reference = set(imputed["unemployment_rate"].index)

for name in PCA_FEATURES:
    dataset_regions = set(features[name].index)
    missing_from_ref = reference - dataset_regions
    level_counts = pd.Index(list(missing_from_ref)).str.len().value_counts().sort_index().to_dict()
    print(f"{name:<20} covers {len(dataset_regions):>4} | missing {len(missing_from_ref):>4} | nuts0={level_counts.get(2,0)} nuts1={level_counts.get(3,0)} nuts2={level_counts.get(4,0)}")

unemployment_rate    covers  509 | missing    0 | nuts0=0 nuts1=0 nuts2=0
poverty_risk         covers  416 | missing  103 | nuts0=0 nuts1=28 nuts2=75
internet_usage       covers  387 | missing  127 | nuts0=0 nuts1=1 nuts2=126
gdp_per_capita       covers  428 | missing   86 | nuts0=3 nuts1=14 nuts2=69
social_trust         covers  509 | missing    0 | nuts0=0 nuts1=0 nuts2=0
life_satisfaction    covers  509 | missing    0 | nuts0=0 nuts1=0 nuts2=0


In [ ]:
def hierarchical_impute(df: pd.DataFrame, reference_index: pd.Index) -> pd.DataFrame:
    """
    For regions in reference_index missing from df:
    1. Try NUTS1 mean (first 3 chars of geo code)
    2. Fall back to NUTS0 mean (first 2 chars)
    3. Fall back to global mean
    """
    # Reindex to full reference — introduces NaN for missing regions
    df = df.reindex(reference_index)

    # Add hierarchy columns temporarily
    df["nuts1"] = df.index.str[:3]
    df["nuts0"] = df.index.str[:2]

    feature_cols = [c for c in df.columns if c not in ["nuts1", "nuts0"]]

    for col in feature_cols:
        # Step 1 — fill with NUTS1 mean
        nuts1_mean = df.groupby("nuts1")[col].transform("mean")
        df[col] = df[col].fillna(nuts1_mean)

        # Step 2 — fill remaining with NUTS0 mean
        nuts0_mean = df.groupby("nuts0")[col].transform("mean")
        df[col] = df[col].fillna(nuts0_mean)

        # Step 3 — fill remaining with global mean
        df[col] = df[col].fillna(df[col].mean())

    df = df.drop(columns=["nuts1", "nuts0"])
    return df


# Apply to datasets with missing regions
for name in ["poverty_risk", "internet_usage", "gdp_per_capita"]:
    before = len(features[name])
    features[name] = hierarchical_impute(features[name], reference_index)
    after  = len(features[name])
    missing = features[name].isnull().sum().sum()
    print(f"✅ {name}: {before} → {after} regions | {missing} missing")

✅ poverty_risk: 416 → 509 regions | 0 missing
✅ internet_usage: 387 → 509 regions | 0 missing
✅ gdp_per_capita: 428 → 509 regions | 0 missing


In [34]:
feature_matrix = pd.concat(features.values(), axis=1, join="inner")

level_counts = feature_matrix.index.str.len().value_counts().sort_index().to_dict()
print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Missing values: {feature_matrix.isnull().sum().sum()}")
print(f"nuts0={level_counts.get(2,0)} nuts1={level_counts.get(3,0)} nuts2={level_counts.get(4,0)}")

Feature matrix shape: (509, 12)
Missing values: 0
nuts0=35 nuts1=123 nuts2=351


In [35]:
# Life expectancy — mean only, used for validation after PCA
le_features = compute_features(imputed["life_expectancy"])
le_features.columns = ["life_expectancy_mean", "life_expectancy_slope"]

# Hierarchical impute to match feature matrix index
le_features = hierarchical_impute(le_features, reference_index)

print(f"life_expectancy: {le_features.shape} | missing: {le_features.isnull().sum().sum()}")
print(f"mean range: [{le_features['life_expectancy_mean'].min():.1f}, {le_features['life_expectancy_mean'].max():.1f}]")

life_expectancy: (509, 2) | missing: 0
mean range: [71.9, 84.6]


In [ ]:
def impute_time_series(df: pd.DataFrame) -> pd.DataFrame:
    """
    Impute missing values within a region's time series:
    1. Linear interpolation for gaps <= 2 consecutive years
    2. Country mean for larger gaps
    3. Forward fill then backward fill for edge gaps
    """
    # Step 1 — interpolate isolated gaps
    df = df.interpolate(axis=1, limit=2, limit_direction="both")

    # Step 2 — country mean for larger gaps
    df["country"] = df.index.str[:2]
    for year in df.columns[:-1]:
        country_mean = df.groupby("country")[year].transform("mean")
        df[year] = df[year].fillna(country_mean)
    df = df.drop(columns=["country"])

    # Step 3 — forward/backward fill for edge gaps
    df = df.ffill(axis=1).bfill(axis=1)

    return df


def impute_missing_regions(df: pd.DataFrame,
                            reference_index: pd.Index) -> pd.DataFrame:
    """
    For regions in reference_index missing from df,
    impute using hierarchical spatial mean:
    1. NUTS1 mean (first 3 chars)
    2. NUTS0 mean (first 2 chars)
    3. Global mean
    """
    df = df.reindex(reference_index)
    df["nuts1"] = df.index.str[:3]
    df["nuts0"] = df.index.str[:2]
    feature_cols = [c for c in df.columns if c not in ["nuts1", "nuts0"]]

    for col in feature_cols:
        nuts1_mean = df.groupby("nuts1")[col].transform("mean")
        df[col]    = df[col].fillna(nuts1_mean)
        nuts0_mean = df.groupby("nuts0")[col].transform("mean")
        df[col]    = df[col].fillna(nuts0_mean)
        df[col]    = df[col].fillna(df[col].mean())

    return df.drop(columns=["nuts1", "nuts0"])


def run_imputation(windowed: dict,
                   pca_features: list,
                   drop_regions: list,
                   reference_dataset: str) -> tuple[dict, pd.Index]:
    """
    Full imputation pipeline:
    1. Time series imputation per dataset
    2. Drop unreliable regions
    3. Get reference index from most complete dataset
    4. Spatial imputation to align all datasets to reference index
    """
    imputed = {}

    # Step 1 — time series imputation
    print("── Step 1: Time series imputation ──")
    for name in pca_features:
        before = windowed[name].isnull().sum().sum()
        imputed[name] = impute_time_series(windowed[name].copy())
        after  = imputed[name].isnull().sum().sum()
        print(f"  {name:<20} {before:>5} missing → {after:>5} missing")

    # Step 2 — drop unreliable regions
    print("\n── Step 2: Drop unreliable regions ──")
    for name in pca_features:
        before = len(imputed[name])
        imputed[name] = imputed[name][~imputed[name].index.isin(drop_regions)]
        after  = len(imputed[name])
        if before != after:
            print(f"  {name:<20} dropped {before - after} regions")

    # Step 3 — reference index from most complete dataset
    reference_index = imputed[reference_dataset].index
    print(f"\n── Step 3: Reference index from '{reference_dataset}' ──")
    print(f"  {len(reference_index)} regions")

    # Step 4 — spatial imputation to align all datasets
    print("\n── Step 4: Spatial imputation ──")
    for name in pca_features:
        before = len(imputed[name])
        imputed[name] = impute_missing_regions(imputed[name], reference_index)
        after  = len(imputed[name])
        missing = imputed[name].isnull().sum().sum()
        print(f"  {name:<20} {before:>4} → {after:>4} regions | {missing} missing")

    return imputed, reference_index

In [37]:
DROP_REGIONS = ["BA", "BA0", "BA00", "XK", "XK0", "XK00"]

imputed, reference_index = run_imputation(
    windowed       = windowed,
    pca_features   = PCA_FEATURES,
    drop_regions   = DROP_REGIONS,
    reference_dataset = "unemployment_rate",
)

── Step 1: Time series imputation ──
  unemployment_rate      598 missing →     0 missing
  poverty_risk          1592 missing →     0 missing
  internet_usage         759 missing →     0 missing
  gdp_per_capita          35 missing →     0 missing
  social_trust            20 missing →     0 missing
  life_satisfaction       19 missing →     0 missing

── Step 2: Drop unreliable regions ──
  unemployment_rate    dropped 1 regions
  poverty_risk         dropped 5 regions
  internet_usage       dropped 2 regions
  social_trust         dropped 1 regions
  life_satisfaction    dropped 1 regions

── Step 3: Reference index from 'unemployment_rate' ──
  509 regions

── Step 4: Spatial imputation ──
  unemployment_rate     509 →  509 regions | 0 missing
  poverty_risk          416 →  509 regions | 0 missing
  internet_usage        387 →  509 regions | 0 missing
  gdp_per_capita        428 →  509 regions | 0 missing
  social_trust           36 →  509 regions | 0 missing
  life_satisfaction   

In [38]:
# Compute mean + slope for all PCA features
features = {}
for name in PCA_FEATURES:
    feat = compute_features(imputed[name])
    feat.columns = [f"{name}_mean", f"{name}_slope"]
    features[name] = feat

# Broadcast country-level to all regions (already done by spatial imputation)
# Just concat directly
feature_matrix = pd.concat(features.values(), axis=1, join="inner")
print(f"Feature matrix: {feature_matrix.shape} | missing: {feature_matrix.isnull().sum().sum()}")

# Compute life expectancy for validation
le_imputed = impute_time_series(windowed["life_expectancy"].copy())
le_imputed = impute_missing_regions(le_imputed, reference_index)
le_features = compute_features(le_imputed)
le_features.columns = ["life_expectancy_mean", "life_expectancy_slope"]
print(f"life_expectancy: {le_features.shape} | missing: {le_features.isnull().sum().sum()}")

Feature matrix: (509, 12) | missing: 0
life_expectancy: (509, 2) | missing: 0


In [39]:
from sklearn.preprocessing import StandardScaler
import joblib

ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Fit scaler on feature matrix
scaler = StandardScaler()
feature_matrix_scaled = pd.DataFrame(
    scaler.fit_transform(feature_matrix),
    index=feature_matrix.index,
    columns=feature_matrix.columns,
)

# Save scaler to artifacts
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")

print(f"✅ Scaler fitted and saved")
print(f"   Input shape: {feature_matrix_scaled.shape}")
print(f"\nMeans (should all be ~0):")
print(feature_matrix_scaled.mean().round(6).to_string())
print(f"\nStds (should all be ~1):")
print(feature_matrix_scaled.std().round(6).to_string())

✅ Scaler fitted and saved
   Input shape: (509, 12)

Means (should all be ~0):
unemployment_rate_mean    -0.0
unemployment_rate_slope    0.0
poverty_risk_mean          0.0
poverty_risk_slope        -0.0
internet_usage_mean       -0.0
internet_usage_slope      -0.0
gdp_per_capita_mean       -0.0
gdp_per_capita_slope       0.0
social_trust_mean         -0.0
social_trust_slope         0.0
life_satisfaction_mean     0.0
life_satisfaction_slope    0.0

Stds (should all be ~1):
unemployment_rate_mean     1.000984
unemployment_rate_slope    1.000984
poverty_risk_mean          1.000984
poverty_risk_slope         1.000984
internet_usage_mean        1.000984
internet_usage_slope       1.000984
gdp_per_capita_mean        1.000984
gdp_per_capita_slope       1.000984
social_trust_mean          1.000984
social_trust_slope         1.000984
life_satisfaction_mean     1.000984
life_satisfaction_slope    1.000984


In [41]:
# Save feature matrix — raw and scaled
feature_matrix.to_parquet(PROCESSED_DIR / "feature_matrix.parquet")
feature_matrix_scaled.to_parquet(PROCESSED_DIR / "feature_matrix_scaled.parquet")

# Save validation dataset
le_features.to_parquet(PROCESSED_DIR / "life_expectancy_validation.parquet")

# Save contextual datasets — mean only, for UI sidebar
CONTEXTUAL = [
    "aerobic_activity", "family_contact", "heavy_drinking",
    "obesity_rate", "smoking", "social_contact", "social_support",
]
for name in CONTEXTUAL:
    if name in windowed:
        df = windowed[name].copy()
        df["mean"] = df.mean(axis=1)
        df.columns = [str(c) for c in df.columns]  # convert all to strings
        df.to_parquet(PROCESSED_DIR / f"{name}_contextual.parquet")

print("✅ Saved:")
print(f"   feature_matrix.parquet          {feature_matrix.shape}")
print(f"   feature_matrix_scaled.parquet   {feature_matrix_scaled.shape}")
print(f"   life_expectancy_validation.parquet {le_features.shape}")
print(f"   contextual datasets: {len(CONTEXTUAL)}")
print("\n🎉 Preprocessing complete")

✅ Saved:
   feature_matrix.parquet          (509, 12)
   feature_matrix_scaled.parquet   (509, 12)
   life_expectancy_validation.parquet (509, 2)
   contextual datasets: 7

🎉 Preprocessing complete


In [4]:
df = pd.read_parquet(INTERIM_DIR / "unemployment_rate.parquet")

nuts2 = df[df.index.str.len() == 4]
print(f"Total NUTS2 regions in unemployment_rate: {len(nuts2)}")
print(f"\nSample:")
print(nuts2.index[:20].tolist())

# Cross-check against NUTS2 from other datasets
for name in ["life_expectancy", "poverty_risk", "gdp_per_capita", "internet_usage"]:
    other = pd.read_parquet(INTERIM_DIR / f"{name}.parquet")
    other_nuts2 = set(other[other.index.str.len() == 4].index)
    unemp_nuts2 = set(nuts2.index)
    in_other_not_unemp = other_nuts2 - unemp_nuts2
    if in_other_not_unemp:
        print(f"\n⚠️  {name} has NUTS2 regions NOT in unemployment_rate:")
        print(sorted(in_other_not_unemp))
    else:
        print(f"✅ {name} — all NUTS2 regions covered by unemployment_rate")

Total NUTS2 regions in unemployment_rate: 351

Sample:
['AT11', 'AT12', 'AT13', 'AT21', 'AT22', 'AT31', 'AT32', 'AT33', 'AT34', 'BE10', 'BE21', 'BE22', 'BE23', 'BE24', 'BE25', 'BE31', 'BE32', 'BE33', 'BE34', 'BE35']

⚠️  life_expectancy has NUTS2 regions NOT in unemployment_rate:
['AL01', 'AL02', 'AL03', 'LI00']

⚠️  poverty_risk has NUTS2 regions NOT in unemployment_rate:
['AL01', 'AL02', 'AL03', 'FI13', 'FI18', 'FI1A', 'XK00']

⚠️  gdp_per_capita has NUTS2 regions NOT in unemployment_rate:
['AL01', 'AL02', 'AL03']
✅ internet_usage — all NUTS2 regions covered by unemployment_rate


In [5]:
df = pd.read_parquet(INTERIM_DIR / "unemployment_rate.parquet")
fi_regions = df[df.index.str.startswith("FI")]
print(fi_regions.index.tolist())

['FI', 'FI1', 'FI19', 'FI1B', 'FI1C', 'FI1D', 'FI2', 'FI20']


In [7]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
NUTS_RECODE  = {"FI13": "FI1D", "FI18": "FI1C", "FI1A": "FI1B"}

for name in ["poverty_risk", "life_expectancy", "unemployment_rate", "gdp_per_capita"]:
    df = pd.read_parquet(INTERIM_DIR / f"{name}.parquet")
    df = df.rename(index=NUTS_RECODE)
    dupes = df.index[df.index.duplicated()].tolist()
    if dupes:
        print(f"{name}: duplicates → {dupes}")
    else:
        print(f"{name}: no duplicates")

poverty_risk: duplicates → ['FI1B', 'FI1C', 'FI1D']
life_expectancy: no duplicates
unemployment_rate: no duplicates
gdp_per_capita: no duplicates
